In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
import torch
from torch.utils.data import Dataset, DataLoader
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt

In [2]:
torch.manual_seed(42)    # we want exact same rows whenever we run the code

In [3]:
df = pd.read_csv("fmnist_small.csv")

In [4]:
df.head()

,label,pixel1,pixel2,pixel3,pixel4,pixel5,pixel6,pixel7,pixel8,pixel9,...,pixel775,pixel776,pixel777,pixel778,pixel779,pixel780,pixel781,pixel782,pixel783,pixel784
0,9,0,0,0,0,0,0,0,0,0,...,0,7,0,50,205,196,213,165,0,0
1,7,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,0,0,0,0,0,0,1,0,0,0,...,142,142,142,21,0,3,0,0,0,0
3,8,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,8,0,0,0,0,0,0,0,0,0,...,213,203,174,151,188,10,0,0,0,0


In [5]:
# split the data into train and test datasets
x = df.iloc[:,1:].values
y = df.iloc[:,0].values

In [6]:
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size= 0.2, random_state=42)   # we choose 42 samples

In [7]:
# Scaling the features ( since the database has images, we will devide the axes with maximum values of 255)
x_train = x_train / 255.0
y_train = y_train / 255.0

In [8]:
# Create customedataset class
class CustomDataset(Dataset):

    def __init__(self, features, labels):
        """it is advisable to make parameters in tensor in constructor so later on easy to use"""
        self.features = torch.tensor(features, dtype = torch.float32)
        self.labels = torch.tensor(labels, dtype = torch.long)

    def __len__(self):
        return len(self.features)

    def __getitem__(self, index):
        return self.features[index], self.labels[index]

In [9]:
# create train_dataset object
train_dataset = CustomDataset(x_train, y_train)

In [10]:
train_dataset[0]

(tensor([0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
         0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
         0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
         0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
         0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
         0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
         0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
         0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
         0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
         0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
         0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
         0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
         0.0000, 0.0000, 0.0

In [11]:
# create test_dataset object
test_dataset = CustomDataset(x_test, y_test)

In [38]:
# Create test and train loader
train_loader = DataLoader(train_dataset, batch_size = 32, shuffle = True)
test_loader = DataLoader(test_dataset, batch_size = 32, shuffle = False)   # It is advisable not to do shuffle on test_dataset


In [28]:
# Define NN Class
class myNN(nn.Module):
    """See the notebook. We have 784 inputs so linear() and will give 128 o/p. Activation fun will be ReLU()"""
    def __init__(self, num_features):

        super().__init__()
        self.model = nn.Sequential(
            nn.Linear(num_features,128),
            nn.ReLU(),
            nn.Linear(128,64),
            nn.ReLU(),
            nn.Linear(64,10)
        )

    def forward(self,x):
        return self.model(x)

In [40]:
# Set Learning rate and epochs
epochs = 100
learning_rate = 0.1

In [41]:
len(train_loader)        # This will say how many batches you have

150

In [33]:
# Instantiate the model
model = myNN(x_train.shape[1])                # will mention how many features(columns in DB) we have

# LOSS FUNCTION
# If you're doing multi-label classification (i.e., each sample can belong to multiple classes), use nn.BCEWithLogitsLoss()
criterion = nn.CrossEntropyLoss()  # Use nn.CrossEntropyLoss() in PyTorch when you're dealing with multi-class classification problems 

# optimizer
optimizer = optim.SGD(model.parameters(), lr=learning_rate)

In [43]:
# training loop
for epoch in range(epochs):
    total_epoch_loss = 0
    
    for batch_features, batch_labels in train_loader:
        #forward pass
        outputs = model(batch_features) 
        
        # Calculate loss
        loss = criterion(outputs, batch_labels)
        
        # optiomize grad
        optimizer.zero_grad()
        
        # back propogation
        loss.backward()

        # update gradients
        optimizer.step()        

        # we will add loss for each run
        total_epoch_loss = total_epoch_loss + loss.item()

    # calculate average loass
    avg_loss = total_epoch_loss / len(train_loader)
    print(f"Epoch: {epoch +1}, Loss: {avg_loss}")

Epoch: 1, Loss: 4.820213479514261e-06
Epoch: 2, Loss: 4.793788736120834e-06
Epoch: 3, Loss: 4.767623016757779e-06
Epoch: 4, Loss: 4.741727074903205e-06
Epoch: 5, Loss: 4.716077637720121e-06
Epoch: 6, Loss: 4.690545047290051e-06
Epoch: 7, Loss: 4.665558000975049e-06
Epoch: 8, Loss: 4.640659225896293e-06
Epoch: 9, Loss: 4.616003510227173e-06
Epoch: 10, Loss: 4.59131351946714e-06
Epoch: 11, Loss: 4.5673753459289185e-06
Epoch: 12, Loss: 4.543704773450526e-06
Epoch: 13, Loss: 4.520288701295172e-06
Epoch: 14, Loss: 4.496702027193914e-06
Epoch: 15, Loss: 4.4733956519227296e-06
Epoch: 16, Loss: 4.450425579263969e-06
Epoch: 17, Loss: 4.427581883453453e-06
Epoch: 18, Loss: 4.405461528107502e-06
Epoch: 19, Loss: 4.383021213219725e-06
Epoch: 20, Loss: 4.360924698194601e-06
Epoch: 21, Loss: 4.3389060805572775e-06
Epoch: 22, Loss: 4.31740100143981e-06
Epoch: 23, Loss: 4.29566459087205e-06
Epoch: 24, Loss: 4.274940654330391e-06
Epoch: 25, Loss: 4.253370352932014e-06
Epoch: 26, Loss: 4.232232338621244

In [44]:
# Evaluate the model now

model.eval()

myNN(
  (model): Sequential(
    (0): Linear(in_features=784, out_features=128, bias=True)
    (1): ReLU()
    (2): Linear(in_features=128, out_features=64, bias=True)
    (3): ReLU()
    (4): Linear(in_features=64, out_features=10, bias=True)
  )
)

In [46]:
# Evaluation code
total = 0
correct = 0
with torch.no_grad():
    
    for batch_features, batch_labels in test_loader:
        
        outputs = model(batch_features)
        
        _, predicted = torch.max(outputs, 1)

        total = total + batch_labels.shape[0]

        correct = correct + (predicted == batch_labels).sum().item()

print(correct/total)

0.12166666666666667


In [45]:
len(test_loader)

38